# ***Predictive Modeling***

To identify beneficiaries at elevated risk of healthcare cost burden, a binary classification model was developed for two outcomes: **high out-of-pocket (OOP) burden** and **high payer cost burden**. 3 modeling approaches—**logistic regression, random forest, and XGBoost**—are evaluated to compare interpretable and nonlinear machine learning methods.

The prediction framework accounts for the information that would realistically be available at each point in time:

* **2008:** Predictions use baseline beneficiary demographics and chronic disease information.
* **2009:** Predictions incorporate 2008 historical information alongside beneficiary characteristics.
* **2010:** Predictions incorporate cumulative information from **2008 and 2009**, allowing the models to leverage a longer history of beneficiary utilization, disease burden, and spending.

This temporal structure is designed to prevent information leakage by ensuring that predictors come from information available **before the outcome being predicted**.

The analysis therefore addresses two related questions:

1. **Can we identify beneficiaries who will experience high OOP or payer cost burden?**
2. **Does incorporating longitudinal beneficiary history improve our ability to identify future high-cost beneficiaries?**

Model performance is evaluated using discrimination and classification metrics, while feature-level interpretation is used to examine which demographic, clinical, utilization, and historical cost factors contribute most to high-cost predictions.


In [1]:
import numpy as np
import pandas as pd
import pickle
import os
import joblib
from src.modeling import *
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
patient = pd.read_csv("cleaned_data/patient_final.csv")
patient.head()

,DESYNPUF_ID,BENE_ESRD_IND,BENE_COUNTY_CD,PART_A_INSUR_CVRAGE_TOT_MONS,PART_B_INSUR_CVRAGE_TOT_MONS,HMO_INSUR_CVRAGE_TOT_MONS,PART_D_INSUR_CVRAGE_TOT_MONS,year,SEX,RACE,STATE,age,age_band,IP_bill,OP_bill,Carrier_bill,num_carrier_claims,avg_carrier_claim_duration,max_carrier_claim_duration,total_carrier_claim_duration,avg_carrier_rejected_amt,max_carrier_rejected_amt,total_carrier_rejected_amt,avg_carrier_claim_rejected_rate (%),max_carrier_claim_rejected_rate (%),avg_carrier_reimb_allowed_diff,min_carrier_reimb_allowed_diff,max_carrier_reimb_allowed_diff,total_carrier_reimb_allowed_diff,max_carrier_chronic_code_counts,avg_carrier_insurance_cost,max_carrier_insurance_cost,total_carrier_insurance_cost,avg_carrier_beneficiary_cost,max_carrier_beneficiary_cost,total_carrier_beneficiary_cost,has_Additional endocrine/metabolic burden,has_COPD,has_Administrative_screening_aftercare,has_Cancer,has_Injury/Frailty Burden,has_Additional cardiovascular burden,has_Additional mental health burden,has_Additional Musculoskeletal Burden,has_Infectious disease burden,has_Stroke/TIA,has_Additional neurologic burden,has_Diabetes,has_Additional renal burden,has_Genitourinary (non-renal) burden,has_Sensory Burden,has_Additional Hematologic burden,has_Rheumatoid Arthritis/Osteoarthritis,has_Depression,has_Heart Failure,has_CKD,has_Additional GI/Hepatic burden,num_ip_claims,avg_ip_claim_duration,max_ip_claim_duration,total_ip_claim_duration,avg_ip_pmt,max_ip_pmt,total_ip_pmt,max_ip_chronic_code_counts,avg_inpatient_insurance_cost,max_inpatient_insurance_cost,total_inpatient_insurance_cost,avg_inpatient_beneficiary_cost,max_inpatient_beneficiary_cost,total_inpatient_beneficiary_cost,num_op_claims,avg_op_claim_duration,max_op_claim_duration,total_op_claim_duration,avg_op_pmt,max_op_pmt,total_op_pmt,max_op_chronic_code_counts,avg_outpatient_insurance_cost,max_outpatient_insurance_cost,total_outpatient_insurance_cost,avg_outpatient_beneficiary_cost,max_outpatient_beneficiary_cost,total_outpatient_beneficiary_cost,num_pde_claims,total_days_medsupplied,avg_days_supplied_per_pdeclaim,max_days_supplied_per_pdeclaim,total_medoop,avg_medoop_per_claim,max_medoop_per_claim,total_raw_drug_cost,avg_raw_drug_cost_per_claim,max_raw_drug_cost_per_claim,has_carrier_record,has_ip_record,has_op_record,has_pde_record,num_service_types,ip_bill_diff,op_bill_diff,total_inpatient_insurance_cost_adj,total_outpatient_insurance_cost_adj,claim_processing_burden,payer_care_cost,out_of_pocket_care_cost,total_financial_impact,High_Payer_Cost,High_OOP_Burden
0,000102649ED5601B,0,290,12,12,0,12,2008,Female,White,VA,77,75-84,0.0,450.0,1030.0,16.0,0 days 01:30:00,1 days,1 days,0.000000,0.0,0.0,0.000000,0.0,22.500000,0.0,210.0,360.0,3.0,63.750000,290.0,1020.0,12.500000,50.0,200.0,1.0,1,1.0,1,1.0,1.0,1.0,1.0,1.0,1,1.0,1,1.0,0.0,0.0,0.0,0,0,0,0,0.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0 days 00:00:00,0 days,0 days,150.000000,320.0,450.0,1.0,120.000000,300.0,360.0,30.000000,70.0,90.0,21.0,760.0,36.190476,90.0,70.0,3.333333,10.0,500.0,23.809524,120.0,True,False,True,True,3.0,0.0,0.0,0.0,360.0,0.0,1880.0,360.0,2240.0,0,0
1,0002278C944E240A,1,250,12,12,0,12,2008,Female,White,NC,72,65-74,0.0,4120.0,6230.0,42.0,0 days 17:08:34.285714285,9 days,30 days,2.142857,50.0,90.0,4.878049,100.0,25.000000,0.0,470.0,1050.0,4.0,130.714280,750.0,5490.0,32.142857,310.0,1350.0,1.0,1,1.0,1,1.0,1.0,1.0,1.0,1.0,0,1.0,1,1.0,1.0,1.0,1.0,1,1,1,1,1.0,0.0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.0,1 days 03:41:32.307692307,13 days,15 days,316.923077,1050.0,4120.0,7.0,199.230769,800.0,2590.0,117.692308,1000.0,1530.0,47.0,1210.0,25.744681,90.0,440.0,9.361702,170.0,3580.0,76.170213,570.0,True,False,True,True,3.0,0.0,0.0,0.0,2590.0,90.0,11660.0,3320.0,14980.0,0,1
2,000330E625C93700,0,420,12,12,0,0,2008,Female,White,AR,75,75-84,0.0,0.0,420.0,4.0,0 days 00:00:00,0 days,0 days,0.000000,0.0,0.0,0.000000,0.0,30.000000,0.0,60.0,120.0,1.0,67.500000,90.0,270.0,37.500000,70.0,150.0,0.0,0,0.0,0,0.0,1.0,0.0,

In [3]:
patient.groupby("year")[
    [
        "payer_care_cost",
        "out_of_pocket_care_cost",
        "total_medoop",
        "total_raw_drug_cost",
        "total_carrier_insurance_cost"
    ]
].agg(["mean", "median", "max"])

payer_care_cost                   out_of_pocket_care_cost         \
                mean  median       max                    mean median   
year                                                                    
2008     5314.272665  1990.0  197740.0              985.040776  490.0   
2009     5687.295894  2750.0  171020.0             1088.280348  670.0   
2010     3331.841783  1670.0  151680.0              653.572111  400.0   

              total_medoop                total_raw_drug_cost                  \
          max         mean median     max                mean median      max   
year                                                                            
2008  42974.0    177.67976   50.0  2490.0         1060.589280  270.0  15690.0   
2009  40902.0    197.86768   80.0  2290.0         1175.498656  440.0  13540.0   
2010  35970.0    126.67884   40.0  1750.0          755.486303  250.0  11150.0   

     total_carrier_insurance_cost                  
                             mean median      max  
year                                               
2008                  1269.084474  670.0  21420.0  
2009                  1456.333748  980.0  19050.0  
2010                   916.378835  590.0  10720.0

In [4]:
patient.groupby("year")[
    ["High_Payer_Cost", "High_OOP_Burden"]
].mean()

,High_Payer_Cost,High_OOP_Burden
year,,
2008,0.099921,0.099776
2009,0.099948,0.099722
2010,0.099939,0.099975


# **Part 1: Logistic Regression**

In [4]:
all_lr_results = []
lr_models = {}

for year in [2008, 2009, 2010]:
    X, y = build_features(patient, year)
    X = fix_feature_dtypes(X)
    results, models = run_logistic_regression_experiment(X, y)
    results["year"] = year
    all_lr_results.append(results)
    
    for target, model in models.items():
        model_path = (
            f"results/models/"
            f"lr_{year}_{target}.joblib"
        )
        joblib.dump(model, model_path)
        lr_models[(year, target)] = model
lr_results_df = pd.concat(all_lr_results, ignore_index=True)
lr_results_df.to_csv("results/lr_results.csv", index=False)

Accuracy: 0.8256
ROC-AUC: 0.6646

Classification Report
              precision    recall  f1-score   support

           0       0.93      0.88      0.90     19964
           1       0.25      0.37      0.30      2216

    accuracy                           0.83     22180
   macro avg       0.59      0.62      0.60     22180
weighted avg       0.86      0.83      0.84     22180


Confusion Matrix
[[17486  2478]
 [ 1390   826]]
Accuracy: 0.8348
ROC-AUC: 0.6709

Classification Report
              precision    recall  f1-score   support

           0       0.93      0.89      0.91     19967
           1       0.27      0.38      0.32      2213

    accuracy                           0.83     22180
   macro avg       0.60      0.63      0.61     22180
weighted avg       0.86      0.83      0.85     22180


Confusion Matrix
[[17672  2295]
 [ 1370   843]]
Accuracy: 0.7391
ROC-AUC: 0.8333

Classification Report
              precision    recall  f1-score   support

           0       0.96  

Historical utilization and cost information substantially improved prediction relative to a demographic baseline, with the strongest predictive performance observed when one year of recent history was available. Adding an additional year of historical information did not yield proportional gains, suggesting that recent beneficiary behavior may provide more predictive value than simply accumulating additional historical measures.

The 2010 claims data also exhibited a substantial downward shift in payer and beneficiary cost distributions despite similar high-cost classification rates, indicating that year-to-year differences in the underlying cost distribution may influence model performance.

# **Part 2: Random Forest**

In [4]:
rf_results = []

for year in [2008, 2009, 2010]:

    X, y = build_features(patient, year)
    X = fix_feature_dtypes(X)

    for target in ["High_Payer_Cost", "High_OOP_Burden"]:

        result = run_random_forest_experiment(
            X,
            y[target]
        )

        rf_results.append({
            "year": year,
            "target": target,
            "accuracy": result["test_results"]["accuracy"],
            "roc_auc": result["test_results"]["roc_auc"],
            "precision": result["test_results"]["precision"],
            "recall": result["test_results"]["recall"],
            "f1": result["test_results"]["f1"],
            "best_params": result["best_params"],
            "cv_roc_auc": result["cv_roc_auc"]
        })

Fitting 3 folds for each of 8 candidates, totalling 24 fits


Exception ignored in: <function ResourceTracker.__del__ at 0x105261bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


Accuracy: 0.7907
ROC-AUC: 0.6852

Classification Report
              precision    recall  f1-score   support

           0       0.93      0.83      0.88     19964
           1       0.22      0.43      0.29      2216

    accuracy                           0.79     22180
   macro avg       0.57      0.63      0.58     22180
weighted avg       0.86      0.79      0.82     22180


Confusion Matrix
[[16586  3378]
 [ 1265   951]]
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Accuracy: 0.7989
ROC-AUC: 0.7038

Classification Report
              precision    recall  f1-score   support

           0       0.93      0.84      0.88     19967
           1       0.24      0.46      0.31      2213

    accuracy                           0.80     22180
   macro avg       0.59      0.65      0.60     22180
weighted avg       0.86      0.80      0.83     22180


Confusion Matrix
[[16706  3261]
 [ 1199  1014]]
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Accuracy: 0.8663

In [3]:
# forgot to save the model in the first run (46 minutes long) before kernel refreshes, hence pasting the results here

rf_results = [
    {
        "year": 2008,
        "target": "High_Payer_Cost",
        "accuracy": 0.7906672678088368,
        "roc_auc": 0.6852472876267535,
        "precision": 0.21968121968121967,
        "recall": 0.42915162454873645,
        "f1": 0.2906035141329259,
        "best_params": {
            "model__max_depth": 20,
            "model__max_features": "sqrt",
            "model__min_samples_leaf": 5,
            "model__n_estimators": 300
        },
        "cv_roc_auc": 0.6920557324848738
    },
    {
        "year": 2008,
        "target": "High_OOP_Burden",
        "accuracy": 0.7989179440937781,
        "roc_auc": 0.7037738341467217,
        "precision": 0.23719298245614034,
        "recall": 0.45820153637596023,
        "f1": 0.312577065351418,
        "best_params": {
            "model__max_depth": 20,
            "model__max_features": "sqrt",
            "model__min_samples_leaf": 5,
            "model__n_estimators": 300
        },
        "cv_roc_auc": 0.6935398483325638
    },
    {
        "year": 2009,
        "target": "High_Payer_Cost",
        "accuracy": 0.8662759242560866,
        "roc_auc": 0.8344435175304352,
        "precision": 0.3727488956846755,
        "recall": 0.4948128101037438,
        "f1": 0.4251937984496124,
        "best_params": {
            "model__max_depth": None,
            "model__max_features": "sqrt",
            "model__min_samples_leaf": 5,
            "model__n_estimators": 300
        },
        "cv_roc_auc": 0.8324217350847141
    },
    {
        "year": 2009,
        "target": "High_OOP_Burden",
        "accuracy": 0.8773219116321009,
        "roc_auc": 0.8983926384384998,
        "precision": 0.42800565770862803,
        "recall": 0.683996383363472,
        "f1": 0.5265355837828433,
        "best_params": {
            "model__max_depth": None,
            "model__max_features": "sqrt",
            "model__min_samples_leaf": 5,
            "model__n_estimators": 300
        },
        "cv_roc_auc": 0.8991162312975559
    },
    {
        "year": 2010,
        "target": "High_Payer_Cost",
        "accuracy": 0.8582957619477006,
        "roc_auc": 0.7851164030994553,
        "precision": 0.32233307751343054,
        "recall": 0.37889039242219213,
        "f1": 0.3483309143686502,
        "best_params": {
            "model__max_depth": None,
            "model__max_features": "sqrt",
            "model__min_samples_leaf": 5,
            "model__n_estimators": 300
        },
        "cv_roc_auc": 0.7828835395618245
    },
    {
        "year": 2010,
        "target": "High_OOP_Burden",
        "accuracy": 0.8434625788999098,
        "roc_auc": 0.823788532917607,
        "precision": 0.31462333825701627,
        "recall": 0.4803788903924222,
        "f1": 0.3802213495180293,
        "best_params": {
            "model__max_depth": None,
            "model__max_features": "sqrt",
            "model__min_samples_leaf": 5,
            "model__n_estimators": 300
        },
        "cv_roc_auc": 0.8243536217357236
    }
]

#rf_results_df = pd.DataFrame(rf_results)

#rf_results_df.to_csv("results/rf_results.csv", index=False)

In [5]:
rf_models = {}

for result in rf_results:

    year = result["year"]
    target = result["target"]

    X, y = build_features(patient, year)
    X = fix_feature_dtypes(X)


    model = refit_random_forest(
        X,
        y[target],
        result["best_params"]
    )

    model_path = (
        f"results/models/"
        f"rf_{year}_{target}.joblib"
    )

    joblib.dump(model, model_path)

    rf_models[(year, target)] = model

    print(f"Saved: {model_path}")

Saved: results/models/rf_2008_High_Payer_Cost.joblib
Saved: results/models/rf_2008_High_OOP_Burden.joblib
Saved: results/models/rf_2009_High_Payer_Cost.joblib
Saved: results/models/rf_2009_High_OOP_Burden.joblib
Saved: results/models/rf_2010_High_Payer_Cost.joblib
Saved: results/models/rf_2010_High_OOP_Burden.joblib


Logistic regression and random forest achieved broadly comparable discrimination, with logistic regression performing particularly well in 2009 and random forest showing modest gains for the 2010 outcomes.

# **Part 3: XGBoost**

In [3]:
xgb_results = []

for year in [2008, 2009, 2010]:

    X, y = build_features(patient, year)
    X = fix_feature_dtypes(X)

    for target in ["High_Payer_Cost", "High_OOP_Burden"]:

        result = run_xgboost_experiment(X, y[target])

        xgb_results.append({
            "year": year,
            "target": target,
            "accuracy": result["accuracy"],
            "roc_auc": result["roc_auc"],
            "precision": result["precision"],
            "recall": result["recall"],
            "f1": result["f1"],
            "cv_roc_auc": result["cv_roc_auc"],
            "best_params": result["best_params"],
            "model": result["model"]
        })

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Accuracy: 0.8999
ROC-AUC: 0.6901

Classification Report
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     19964
           1       0.48      0.03      0.05      2216

    accuracy                           0.90     22180
   macro avg       0.69      0.51      0.50     22180
weighted avg       0.86      0.90      0.86     22180


Confusion Matrix
[[19896    68]
 [ 2152    64]]
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Accuracy: 0.9013
ROC-AUC: 0.7054

Classification Report
              precision    recall  f1-score   support

           0       0.90      1.00      0.95     19967
           1       0.61      0.03      0.06      2213

    accuracy                           0.90     22180
   macro avg       0.75      0.51      0.50     22180
weighted avg       0.87      0.90      0.86     22180


Confusion Matrix
[[19924    43]
 [ 2147    66]]
Fitting 3 folds 

In [12]:
import joblib

os.makedirs("results/models", exist_ok=True)

for result in xgb_results:
    model_path = (
        f"results/models/"
        f"xgb_{result['year']}_{result['target']}.joblib"
    )

    joblib.dump(result["model"], model_path)
    result["model_path"] = model_path

xgb_results_df = pd.DataFrame(xgb_results)

xgb_results_df.to_csv(
    "results/xgb_results.csv",
    index=False
)

**Model performance comparison**

Across the six prediction settings, all three modeling approaches demonstrate meaningful predictive signal, but their relative performance varies substantially by prediction year and target. Overall, XGBoost achieves the highest ROC-AUC in all six settings, although its advantage over logistic regression and random forest is generally modest. 

In particular, XGBoost's improvements over logistic regression are largest in the weaker 2008 settings, but even there the absolute improvement is only approximately 0.02–0.03 ROC-AUC. In the strongest 2009 settings, the difference is only around 0.003–0.005 ROC-AUC. This suggests that feature construction may be more consequential than model complexity for this task. Rather than requiring a highly nonlinear model to uncover hidden patterns, the engineered features appear to have transformed raw claims information into clinically and operationally meaningful predictors that are already highly informative in a relatively simple linear model.

The results also show a consistent difference between the two prediction targets. High OOP Burden is generally easier to predict than High Payer Cost, particularly in 2009. This may reflect the fact that beneficiary-level out-of-pocket burden is more directly related to observable prior utilization and patient cost characteristics, whereas payer cost may be influenced by additional factors that are less directly captured by the available features.

# **Part 4: Threshold analysis**

# **Part 5: Feature interpretation**
- LR: coefficients/odds ratios
- RF: feature importance + SHAP
- XGBoost: feature importance + SHAP

# **Part 6: Ablation study**

compare 1) baseline 2) + clinical history, 3) + utilization, 4) + cost, 5) + risk history
see how much each data adds to the predictive analysis